##### Step 1: Setup and Importing Libraries
First, we need to import all the necessary tools (libraries) for our project. It's like gathering your ingredients before you start cooking.

Markdown Explanation
os, dotenv: These are used to manage secret information, specifically your API key. dotenv helps load it from a local .env file, which is a good practice to keep your keys safe and out of the main code.

re: This is the "Regular Expressions" library. It's a powerful tool for finding patterns in text. We'll use it to parse the AI's response and separate the questions, options, and answers.

json: This library is for working with JSON data, which is a simple and clean format for storing data. We'll use it to save and load our quiz history.

pathlib: Makes working with file paths (like quiz_history.json) easier and more reliable across different operating systems (Windows, macOS, Linux).

datetime: Used to get the current date and time, which we'll use to timestamp our saved quizzes.

langchain_groq, langchain_core: These are parts of the LangChain library. They provide the tools to connect to the Groq API, create a prompt "template," and build a "chain" to process our request.

In [2]:
! pip install -r requirements.txt


ERROR: Could not open requirements file: [Errno 2] No such file or directory: 'requirements.txt'
You should consider upgrading via the 'C:\Users\kushagra\Desktop\CSAI\my-elysium-app\venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [3]:
# First, make sure you have the required libraries installed:
# pip install langchain-groq python-dotenv streamlit

import os, re, json
from pathlib import Path
from datetime import datetime
from typing import List, Dict

from dotenv import load_dotenv
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage

# This line loads the environment variables from a file named '.env'
# Create a file named .env in the same directory and add:
# GROQ_API_KEY="your_api_key_here"
load_dotenv()

print("Libraries imported successfully!")

c:\Users\kushagra\Desktop\CSAI\my-elysium-app\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Libraries imported successfully!


The get_llm function does this job.

It first tries to get the GROQ_API_KEY that you saved in your .env file.

If the key is found, it creates a ChatGroq object. This object is our actual connection to the LLM.

We pass the model_name (like "llama-3.1-8b-instant") and a temperature setting. Temperature controls the randomness of the AI's output. 0.7 is a good balance between creative and predictable.

In the original script, @st.cache_resource is a special Streamlit command to prevent re-creating this connection every time a user clicks a button. In our notebook, we'll just call the function directly.

In [4]:
def get_llm(model_name: str):
    """Initializes and returns the ChatGroq LLM object."""
    api_key = os.getenv("GROQ_API_KEY")
    if not api_key:
        print("🚨 GROQ_API_KEY not found! Please set it in your .env file.")
        return None
    return ChatGroq(groq_api_key=api_key, model_name=model_name, temperature=0.7)

# Let's create our LLM object
llm = get_llm("llama-3.1-8b-instant")

if llm:
    print("LLM object created successfully.")
    print(llm) # You can uncomment this to see the object details

LLM object created successfully.
client=<groq.resources.chat.completions.Completions object at 0x000001860D3FD330> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001860D49D7B0> model_name='llama-3.1-8b-instant' model_kwargs={} groq_api_key=SecretStr('**********')



#### Step 3: Creating the Prompt Template
We can't just send our topic to the AI. We need to give it clear instructions on what to do. A prompt template is like a recipe or a form letter that we fill out before sending it.

ChatPromptTemplate helps us structure our instructions.


System Message: This sets the "persona" or role for the AI. We're telling it, "You are an expert exam question setter." This helps it generate better, more relevant content.


User Message: This is our actual request. It contains placeholders {topic} and {num_questions}. LangChain will automatically replace these with the actual topic and number we provide later.

Instructions: We give it a numbered list of rules to follow to ensure the output format is consistent and easy for our program to read.

In [ ]:
mcq_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an expert exam question setter. Generate multiple-choice questions (MCQs) on the given topic."),
    ("user", """Topic: {topic}
Number of Questions: {num_questions}

Instructions:
1. Each question must be clear and concise.
2. Provide 4 options (a, b, c, d).
3. Only one option should be correct.
4. After each question, include: Answer: <a|b|c|d>
5. Number questions like: 1., 2., 3., ...
""")
])

print("Prompt template created.")
# print(mcq_prompt) # You can uncomment this to see the template details

Prompt template created.



#### Step 4: Building and Running the "Chain"
In LangChain, a "chain" connects our components together. It defines a sequence of steps. Our chain is simple: Prompt -> LLM -> Output Parser.

Markdown Explanation
mcq_prompt: The template we just defined.

llm: The connection to the Groq model.

StrOutputParser(): The AI model's response is actually a complex object. This parser simplifies it by just pulling out the text content, giving us a clean string to work with.

The | symbol (pipe) is a cool LangChain feature to link these components together.

chain.invoke(...): This is the command to actually run the chain. We provide the values for our placeholders (topic and num_questions).

In [6]:
!pip install --upgrade langchain-groq groq

You should consider upgrading via the 'C:\Users\kushagra\Desktop\CSAI\my-elysium-app\venv\Scripts\python.exe -m pip install --upgrade pip' command.


In [7]:
#only build the chain if llm is created
if llm:
    #Build the chain
    chain =(mcq_prompt | llm | StrOutputParser())
    print("Chain build succesfully")

    #Now , let's run the chain with an example 
    print("\n-- Invoking Chain ---")
    topic="The Solar System"
    num_questions=3

    ## this is the part that actually run the LLM
    raw_response=chain.invoke({"topic" :topic,"num_questions":num_questions})
    print("\n-- Raw Response ---")
    print(raw_response)

else:
    print("Cannot created chain as LLM is not initialised.")
    raw_response="" #Define it as empty for the next step

Chain build succesfully

-- Invoking Chain ---

-- Raw Response ---
1. What is the largest planet in our Solar System?
a. Earth
b. Saturn
c. Jupiter
d. Uranus

Answer: c

2. Which of the following planets is known for being the hottest in the Solar System?
a. Mercury
b. Mars
c. Venus
d. Jupiter

Answer: c

3. Which of the following moons is orbiting the planet Saturn?
a. Titan of Jupiter
b. Europa of Jupiter
c. Titan of Saturn
d. Ganymede of Jupiter

Answer: c


### Step 5: Parsing the AI's Response
The raw_response we got is just a single block of text. For our app to work with it (e.g., check answers), we need to break it down into a structured format, like a list of questions, where each question has its options and the correct answer. This is where we use Regular Expressions (re).

Markdown Explanation
The parse_mcqs function is a bit complex, but here's the breakdown:

re.split(r'(?m)^(?=\d+\.\s)', text.strip()): This is the key part. It splits the entire text block into a list, with each item starting with a number like 1., 2., etc. This separates the questions.

It then loops through each question block.

Inside the loop, it finds the question line, the option lines (a, b, c, d), and the answer line.

It uses other small regular expressions to extract just the option letter (a), the option text, and the correct answer letter.

Finally, it puts everything into a nice Python dictionary {"question": ..., "options": [...], "answer": ...} and adds it to a list.



In [8]:
def parse_mcqs(text: str) -> List[Dict]:
    """Return list of dicts: {question, options(['a) ...']), answer('a'|'b'|'c'|'d')}"""
    # Split the text into blocks for each question. The regex looks for a line starting with a number and a dot (e.g., "1.").
    parts = re.split(r'(?m)^(?=\d+\.\s)', text.strip())
    out = []
    for block in parts:
        block = block.strip()
        if not block:
            continue
        lines = [l.strip() for l in block.splitlines() if l.strip()]
        if not lines:
            continue

        # The first line is assumed to be the question
        q_line = re.sub(r'^\d+\.\s*', '', lines[0])

        # Find all lines that look like options (e.g., "a) some text")
        opt_pat = re.compile(r'^([a-dA-D])[\.\)]\s+(.*)$')
        options_raw = []
        for l in lines[1:]:
            m = opt_pat.match(l)
            if m:
                letter = m.group(1).lower()
                text_only = m.group(2).strip()
                options_raw.append(f"{letter}) {text_only}")

        # Find the line that contains the answer
        ans = ""
        for l in lines:
            m = re.search(r'answer:\s*([a-dA-D])\b', l, re.IGNORECASE)
            if m:
                ans = m.group(1).lower()
                break

        if q_line and options_raw:
            out.append({"question": q_line, "options": options_raw[:4], "answer": ans})
    return out

# Let's parse the response we got in the previous step
if raw_response:
    parsed_mcqs = parse_mcqs(raw_response)

    print("\n--- Parsed MCQ Data ---")
    # Use json.dumps for a pretty print
    print(json.dumps(parsed_mcqs, indent=2))
else:
    print("No response to parse.")


--- Parsed MCQ Data ---
[
  {
    "question": "What is the largest planet in our Solar System?",
    "options": [
      "a) Earth",
      "b) Saturn",
      "c) Jupiter",
      "d) Uranus"
    ],
    "answer": "c"
  },
  {
    "question": "Which of the following planets is known for being the hottest in the Solar System?",
    "options": [
      "a) Mercury",
      "b) Mars",
      "c) Venus",
      "d) Jupiter"
    ],
    "answer": "c"
  },
  {
    "question": "Which of the following moons is orbiting the planet Saturn?",
    "options": [
      "a) Titan of Jupiter",
      "b) Europa of Jupiter",
      "c) Titan of Saturn",
      "d) Ganymede of Jupiter"
    ],
    "answer": "c"
  }
]


Saving and Loading History
A good app remembers things. Our script saves every generated quiz to a file called quiz_history.json. This is called persistence.

Markdown Explanation
HISTORY_FILE = Path("quiz_history.json"): Defines the name of our history file.

load_history(): This function checks if the file exists. If it does, it reads the content and uses json.loads() to convert the text back into a Python list of dictionaries.

save_history(): This function takes a list of quiz items and uses json.dumps() to convert it into a text string, which it then writes to the file. indent=2 makes the JSON file nicely formatted and human-readable.

add_to_history(): This function simply adds a new quiz record to the existing history list and then calls save_history().

In [9]:
HISTORY_FILE = Path("quiz_history.json")

def load_history() -> List[Dict]:
    """Loads the history from the JSON file."""
    if HISTORY_FILE.exists():
        try:
            return json.loads(HISTORY_FILE.read_text(encoding="utf-8"))
        except Exception as e:
            print(f"Error loading history: {e}")
            return []
    return []

def save_history(items: List[Dict]):
    """Saves the history to the JSON file."""
    HISTORY_FILE.write_text(json.dumps(items, indent=2, ensure_ascii=False), encoding="utf-8")
    print("History saved to quiz_history.json")

def add_to_history(history_list: list, topic: str, num: int, response: str):
    """Adds a new entry to the history list and saves it."""
    new_id = (history_list[-1]["id"] + 1) if history_list else 1
    history_list.append({
        "id": new_id,
        "ts": datetime.now().strftime("%Y-%m-%d %H:%M"),
        "topic": topic,
        "num": int(num),
        "response": response
    })
    save_history(history_list)
    return history_list

# Let's simulate saving our generated quiz to the history
print("\n--- Testing History ---")
# First, load any existing history
current_history = load_history()
print(f"Loaded {len(current_history)} items from history.")

# Now, add our new quiz
if raw_response:
    current_history = add_to_history(current_history, topic, num_questions, raw_response)
    print(f"Added new item. History now has {len(current_history)} items.")

# You can now check for a 'quiz_history.json' file in your directory!


--- Testing History ---
Loaded 0 items from history.
History saved to quiz_history.json
Added new item. History now has 1 items.


In [10]:
# Agar aapke paas 'parsed_mcqs' variable nahi hai, to test ke liye aap yeh sample data use kar sakte hain:
# parsed_mcqs = [
#   {
#     "question": "How do you create an empty dictionary in Python?",
#     "options": ["a) {}", "b) []", "c) ()", "d) new Dictionary()"],
#     "answer": "a"
#   },
#   {
#     "question": "Which method is used to remove a key-value pair from a dictionary?",
#     "options": ["a) remove()", "b) delete()", "c) pop()", "d) discard()"],
#     "answer": "c"
#   }
# ]

print("\n---  interactive Quiz Simulation ---")
print("Let's start the quiz! Please type a, b, c, or d for each question.")

score = 0
total_questions = len(parsed_mcqs)

# Har ek question ke liye loop chalayenge
for i, mcq in enumerate(parsed_mcqs):
    print("\n" + "="*30)
    print(f"Q{i+1}: {mcq['question']}")
    print("-" * 30)
    
    # Options print karenge
    for opt in mcq['options']:
        print(opt)
        
    # User se input lenge
    user_answer = input("Your answer: ").lower().strip()
    
    # Answer check karenge
    if user_answer == mcq['answer']:
        print("\n✅ Correct! Great job.")
        score += 1
    else:
        print(f"\n❌ Wrong! The correct answer was '{mcq['answer']}'.")

# Final score display karenge
print("\n" + "="*30)
print("🎉 Quiz Finished! 🎉")
print(f"Your Final Score: {score} out of {total_questions}")
print("="*30)


---  interactive Quiz Simulation ---
Let's start the quiz! Please type a, b, c, or d for each question.

Q1: What is the largest planet in our Solar System?
------------------------------
a) Earth
b) Saturn
c) Jupiter
d) Uranus


KeyboardInterrupt: Interrupted by user

In [ ]:
c